# Quick tour: RL for Spatial Decision Making

This notebook demonstrates each simulation at a small scale so you can explore the dynamics interactively. For the full reproduction at publication quality, run `python -m src.reproduce_all` from the repo root instead.

**Prerequisite:** Install the repo from its root with `pip install -e .` before running this notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.utils import set_seed
from src.simulations import (
    sim01_qlearning_gridworld as s1,
    sim02_multi_agent as s2,
    sim03_adaptive_sensor as s3,
    sim04_dynamic_obstacle as s4,
)

## 1. Q-learning in a 12×12 gridworld

Train a tabular Q-learning agent for 300 episodes and plot the return curve.

In [ ]:
set_seed(42)
env = s1.GridWorld.default()
Q, returns, steps = s1.train_qlearning(env, n_episodes=300)
print(f'Final avg return (last 50 ep): {np.mean(returns[-50:]):.2f}')
print(f'Final avg steps  (last 50 ep): {np.mean(steps[-50:]):.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(returns, alpha=0.4, label='per-episode')
w = 20
ma = np.convolve(returns, np.ones(w)/w, mode='valid')
ax.plot(range(len(ma)), ma, linewidth=2, label=f'{w}-ep moving avg')
ax.set_xlabel('Episode'); ax.set_ylabel('Cumulative reward'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 2. Multi-agent coordination

Train two independent Q-learners with collision penalties.

In [ ]:
set_seed(7)
env2 = s2.MultiAgentGrid.default()
Q2, joint, te, tl = s2.train_independent_qlearners(env2, n_episodes=400)
print(f'Final joint return (last 50 ep): {np.mean(joint[-50:]):.2f}')

plt.figure(figsize=(8, 4))
plt.plot(joint, alpha=0.4)
w = 20
ma = np.convolve(joint, np.ones(w)/w, mode='valid')
plt.plot(range(len(ma)), ma, linewidth=2)
plt.xlabel('Episode'); plt.ylabel('Joint cumulative reward'); plt.grid(alpha=0.3)
plt.show()

## 3. Adaptive sensor placement

Compare three acquisition strategies on a small pollution field.

In [ ]:
set_seed(3)
field = s3.build_field(n=20)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, strat in zip(axes, ['random', 'greedy', 'rl']):
    np.random.seed(0)
    placements, rmses = s3.run_strategy(strat, field, budget=10, n=20)
    ax.plot(rmses, '-o', markersize=4, label=strat.upper())
    ax.set_title(f'{strat.upper()}\nfinal RMSE = {rmses[-1]:.3f}')
    ax.set_xlabel('Sensors deployed'); ax.set_ylabel('RMSE'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Dynamic obstacle avoidance

Train SARSA to avoid a patrolling obstacle. Uses fewer episodes than the manuscript figure; success rate is lower as a result.

In [ ]:
set_seed(11)
env4 = s4.DynamicObstacleEnv()
Q4, success = s4.train_sarsa(env4, n_episodes=1500)

plt.figure(figsize=(8, 4))
plt.plot(success, linewidth=2)
plt.axhline(0.95, color='green', linestyle='--', label='95% target')
plt.xlabel('Episode'); plt.ylabel('Rolling success rate')
plt.ylim(-0.05, 1.05); plt.grid(alpha=0.3); plt.legend()
plt.show()